# Phase 1: TD(λ) Value Learning — Random Policy

**Goal**: Learn $V_i(s)$ for each agent under random policy, validate against MC ground truth.
Sliding window TD(λ) with pick sub-states (variable γ).

In [7]:
# =============================================================================
# CONFIG (SLURM-injectable via sed/papermill)
# =============================================================================

R_PICKER = -1
LAMBDA = 0
WINDOW_SIZE = 1          # buffer n for TD(λ). λ=0 → effectively 1-step
MLP_LAYERS = [16, 16, 16, 16]

LR = 0.001
LR_MIN = 1e-6
LR_DECAY_FRAC = 0.8         # linear decay over this fraction of training

TRAIN_TICKS = 400_000     # total agent steps
EVAL_FREQ = 1000         # evaluate every N ticks
CHECKPOINT_FREQ = 100_000   # save model every N ticks
PRINT_FREQ = 10_000        # print progress every N ticks

SEED = 42
MC_DIR = "monte_carlo_script/mc_data"
OUTPUT_PATH = "outputs"
TAG = f"td_lam{LAMBDA}_lr{LR}_w{WINDOW_SIZE}_rp{R_PICKER}"

In [8]:
# =============================================================================
# IMPORTS + DERIVED CONSTANTS
# =============================================================================

import os, sys, json, time
import numpy as np
import pandas as pd
import torch
import time

from config import (
    H, W, NUM_AGENTS, GAMMA, GAMMA_STEP, K_NEAREST, INPUT_DIM,
    P_SPAWN, P_DESPAWN, L,
)
from helpers import (
    State, encode_state, ValueNetwork, SlidingWindowBuffer,
    env_step, spawn_despawn, random_action, random_initial_state,
    evaluate_on_mc, get_memory_mb, get_weight_stats,
)
start_time = time.time()
R_OTHER = (1.0 - R_PICKER) / (NUM_AGENTS - 1)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")
print(f"γ_step = {GAMMA_STEP:.6f}, input_dim = {INPUT_DIM}")
print(f"P_spawn = {P_SPAWN:.4f}, P_despawn = {P_DESPAWN:.4f}")
print(f"R_picker = {R_PICKER}, R_other = {R_OTHER:.4f}")
print(f"L = {L} (spawn every {L * NUM_AGENTS} ticks)")

Device: cpu
γ_step = 0.974004, input_dim = 52
P_spawn = 0.0400, P_despawn = 0.1200
R_picker = -1, R_other = 0.6667
L = 1 (spawn every 4 ticks)


In [9]:
# =============================================================================
# LOAD MC GROUND TRUTH
# =============================================================================

mc_files = sorted([f for f in os.listdir(MC_DIR) if f.endswith('.npz')])
print(f"Found {len(mc_files)} MC state files in {MC_DIR}")

mc_states = []
mc_mean_values = []

for f in mc_files:
    data = np.load(os.path.join(MC_DIR, f), allow_pickle=True)
    state = State(
        agent_positions={i: tuple(data['agent_positions'][i]) for i in range(NUM_AGENTS)},
        apple_positions={tuple(p) for p in data['apple_positions']},
        actor=int(data['actor']),
        tick_in_round=int(data['tick_in_round']),
        ticks_until_spawn=int(data['ticks_until_spawn']),
    )
    mc_states.append(state)
    mc_mean_values.append(data['mean_values'])

mc_mean_values = np.array(mc_mean_values)  # (num_states, NUM_AGENTS)

# Summary stats
avg_mag = np.mean(np.abs(mc_mean_values), axis=0)
print(f"MC states: {len(mc_states)}")
for i in range(NUM_AGENTS):
    vals = mc_mean_values[:, i]
    print(f"  Agent {i}: mean={vals.mean():.4f}, std={vals.std():.4f}, "
          f"avg|V|={avg_mag[i]:.4f}")

Found 10 MC state files in monte_carlo_script/mc_data
MC states: 10
  Agent 0: mean=1.2810, std=0.3842, avg|V|=1.2810
  Agent 1: mean=1.0764, std=0.3882, avg|V|=1.0764
  Agent 2: mean=1.0972, std=0.3451, avg|V|=1.0972
  Agent 3: mean=1.2371, std=0.3003, avg|V|=1.2371


In [10]:
# =============================================================================
# CREATE MODELS (one per agent)
# =============================================================================

torch.manual_seed(SEED)
np.random.seed(SEED)

models = []
optimizers = []
for i in range(NUM_AGENTS):
    m = ValueNetwork(INPUT_DIM, MLP_LAYERS).to(device)
    models.append(m)
    optimizers.append(torch.optim.SGD(m.parameters(), lr=LR))

total_params = sum(p.numel() for p in models[0].parameters())
print(f"Model: {MLP_LAYERS}, {total_params} params per agent")
print(models[0])

Model: [16, 16, 16, 16], 1681 params per agent
ValueNetwork(
  (net): Sequential(
    (0): Linear(in_features=52, out_features=16, bias=True)
    (1): LeakyReLU(negative_slope=0.01)
    (2): Linear(in_features=16, out_features=16, bias=True)
    (3): LeakyReLU(negative_slope=0.01)
    (4): Linear(in_features=16, out_features=16, bias=True)
    (5): LeakyReLU(negative_slope=0.01)
    (6): Linear(in_features=16, out_features=16, bias=True)
    (7): LeakyReLU(negative_slope=0.01)
    (8): Linear(in_features=16, out_features=1, bias=True)
  )
)


In [11]:
# =============================================================================
# TRAINING LOOP
# =============================================================================

os.makedirs(OUTPUT_PATH, exist_ok=True)
csv_path = os.path.join(OUTPUT_PATH, f"phase1_{TAG}.csv")
meta_path = csv_path.replace('.csv', '_metadata.json')
ckpt_dir = os.path.join(OUTPUT_PATH, f"checkpoints_{TAG}")
os.makedirs(ckpt_dir, exist_ok=True)

# Metadata
metadata = {
    "phase": 1, "R_PICKER": R_PICKER, "R_OTHER": R_OTHER,
    "LAMBDA": LAMBDA, "WINDOW_SIZE": WINDOW_SIZE,
    "MLP_LAYERS": MLP_LAYERS, "LR": LR, "LR_MIN": LR_MIN,
    "LR_DECAY_FRAC": LR_DECAY_FRAC,
    "TRAIN_TICKS": TRAIN_TICKS, "EVAL_FREQ": EVAL_FREQ,
    "SEED": SEED, "H": H, "W": W, "NUM_AGENTS": NUM_AGENTS,
    "GAMMA": GAMMA, "GAMMA_STEP": GAMMA_STEP,
    "K_NEAREST": K_NEAREST, "INPUT_DIM": INPUT_DIM, "L": L,
    "P_SPAWN": P_SPAWN, "P_DESPAWN": P_DESPAWN,
    "avg_magnitude_true_value": {
        f"agent_{i}": float(np.mean(np.abs(mc_mean_values[:, i])))
        for i in range(NUM_AGENTS)
    },
}
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=4)

# LR schedule: linear decay
def get_lr(tick):
    decay_end = int(TRAIN_TICKS * LR_DECAY_FRAC)
    if tick >= decay_end:
        return LR_MIN
    return LR + (LR_MIN - LR) * tick / decay_end

# Initialize
train_rng = np.random.default_rng(SEED + 1)
state = random_initial_state(np.random.default_rng(SEED + 5000))
buffer = SlidingWindowBuffer(WINDOW_SIZE, NUM_AGENTS)

# First state into buffer
enc = {i: encode_state(state, i) for i in range(NUM_AGENTS)}
buffer.add_state(enc)

first_write = True
prev_weights = {i: [] for i in range(NUM_AGENTS)}
total_grad_steps = 0
t0 = time.time()

print(f"Training: {TRAIN_TICKS} ticks, λ={LAMBDA}, n={WINDOW_SIZE}")
print(f"Logging to: {csv_path}")

for tick in range(TRAIN_TICKS):
    # LR update
    current_lr = get_lr(tick)
    for opt in optimizers:
        for pg in opt.param_groups:
            pg['lr'] = current_lr

    # --- Environment step ---
    actor = state.actor
    new_pos = random_action(state.agent_positions[actor], train_rng)
    new_state, rewards, picked, arrive_state = env_step(
        state, new_pos, R_PICKER, R_OTHER
    )

    if picked:
        # Transition: state → arrive (γ=γ_step, r=0)
        buffer.add_transition({i: 0.0 for i in range(NUM_AGENTS)}, GAMMA_STEP)
        enc_arr = {i: encode_state(arrive_state, i) for i in range(NUM_AGENTS)}
        buffer.add_state(enc_arr)

        # Transition: arrive → new_state (γ=1, r=rewards)
        buffer.add_transition(rewards, 1.0)
    else:
        # Transition: state → new_state (γ=γ_step, r=0)
        buffer.add_transition(rewards, GAMMA_STEP)

    # --- Spawn/despawn BEFORE adding to buffer ---
    if new_state.ticks_until_spawn < 0:
        new_state = spawn_despawn(new_state, train_rng)

    # New state into buffer (post-spawn if applicable)
    enc_new = {i: encode_state(new_state, i) for i in range(NUM_AGENTS)}
    buffer.add_state(enc_new)

    # --- Train on oldest ready state ---
    while buffer.can_train():
        enc_0, targets = buffer.pop_oldest(LAMBDA, models, device)
        for i in range(NUM_AGENTS):
            x = torch.tensor(enc_0[i], dtype=torch.float32, device=device)
            pred = models[i](x)
            loss = (pred - targets[i]) ** 2
            optimizers[i].zero_grad()
            loss.backward()
            optimizers[i].step()
        total_grad_steps += 1

    state = new_state

    # --- Evaluate ---
    if (tick + 1) % EVAL_FREQ == 0:
        metrics = evaluate_on_mc(models, mc_states, mc_mean_values, device)
        row = {'tick': tick + 1, 'total_grad_steps': total_grad_steps,
               'current_lr': current_lr, 'memory_mb': get_memory_mb()}
        row.update(metrics)

        for i in range(NUM_AGENTS):
            ws, prev_weights[i] = get_weight_stats(models[i], prev_weights[i])
            for k, v in ws.items():
                row[f'a{i}_{k}'] = v

        pd.DataFrame([row]).to_csv(
            csv_path, mode='w' if first_write else 'a',
            header=first_write, index=False,
        )
        first_write = False

    # --- Print ---
    if (tick + 1) % PRINT_FREQ == 0:
        elapsed = time.time() - t0
        metrics = evaluate_on_mc(models, mc_states, mc_mean_values, device)
        print(f"Tick {tick+1:>8d} | grad_steps={total_grad_steps} | "
              f"pct_err={metrics['pct_err_total']:.2f}% | "
              f"mae={metrics['mae_total']:.4f} | "
              f"bias={metrics['bias_total']:.4f} | "
              f"lr={current_lr:.2e} | {elapsed:.0f}s")

    # --- Checkpoint ---
    if (tick + 1) % CHECKPOINT_FREQ == 0:
        for i in range(NUM_AGENTS):
            torch.save(models[i].state_dict(),
                       os.path.join(ckpt_dir, f"agent{i}_tick{tick+1}.pt"))

elapsed = time.time() - t0
print(f"\nDone in {elapsed:.1f}s ({total_grad_steps} grad steps)")

Training: 400000 ticks, λ=0, n=1
Logging to: outputs/phase1_td_lam0_lr0.001_w1_rp-1.csv
Tick    10000 | grad_steps=11128 | pct_err=36.12% | mae=0.4235 | bias=-0.0966 | lr=9.69e-04 | 39s
Tick    20000 | grad_steps=22240 | pct_err=25.12% | mae=0.2935 | bias=-0.0578 | lr=9.38e-04 | 76s
Tick    30000 | grad_steps=33331 | pct_err=25.53% | mae=0.2991 | bias=-0.0712 | lr=9.06e-04 | 114s
Tick    40000 | grad_steps=44478 | pct_err=28.93% | mae=0.3412 | bias=-0.0572 | lr=8.75e-04 | 151s
Tick    50000 | grad_steps=55547 | pct_err=30.10% | mae=0.3510 | bias=-0.2302 | lr=8.44e-04 | 188s
Tick    60000 | grad_steps=66677 | pct_err=24.89% | mae=0.2911 | bias=-0.0680 | lr=8.13e-04 | 225s
Tick    70000 | grad_steps=77880 | pct_err=28.30% | mae=0.3300 | bias=-0.1099 | lr=7.81e-04 | 263s
Tick    80000 | grad_steps=88986 | pct_err=26.98% | mae=0.3176 | bias=-0.0760 | lr=7.50e-04 | 300s
Tick    90000 | grad_steps=100125 | pct_err=23.84% | mae=0.2763 | bias=-0.0219 | lr=7.19e-04 | 338s
Tick   100000 | grad_s

In [12]:
# =============================================================================
# FINAL EVALUATION
# =============================================================================
end_time = time.time()
time_taken = end_time - start_time
time_min = time_taken / 60
print(f"Total time taken: {time_taken:.1f}s")
metadata['total_time_seconds'] = time_taken
metadata['total_time_minutes'] = time_min

with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=4)

print("=" * 60)
print("FINAL RESULTS")
print("=" * 60)

metrics = evaluate_on_mc(models, mc_states, mc_mean_values, device)

for i in range(NUM_AGENTS):
    print(f"\nAgent {i}:")
    print(f"  MAE:       {metrics[f'a{i}_mae']:.6f}")
    print(f"  Bias:      {metrics[f'a{i}_bias']:.6f}")
    print(f"  % Error:   {metrics[f'a{i}_pct_error']:.2f}%")
    print(f"  Mean Pred: {metrics[f'a{i}_mean_pred']:.6f}")
    print(f"  Mean True: {metrics[f'a{i}_mean_true']:.6f}")

print(f"\nOverall MAE:     {metrics['mae_total']:.6f}")
print(f"Overall % Error: {metrics['pct_err_total']:.2f}%")

if metrics['pct_err_total'] < 10.0:
    print("*** PASS: <10% error ***")
else:
    print("*** FAIL: >=10% error ***")

# Save final models
for i in range(NUM_AGENTS):
    torch.save(models[i].state_dict(),
               os.path.join(OUTPUT_PATH, f"final_agent{i}_{TAG}.pt"))
print(f"\nFinal models saved to {OUTPUT_PATH}/")

Total time taken: 1489.3s
FINAL RESULTS

Agent 0:
  MAE:       0.388023
  Bias:      -0.257675
  % Error:   30.29%
  Mean Pred: 1.023307
  Mean True: 1.280982

Agent 1:
  MAE:       0.324425
  Bias:      -0.001045
  % Error:   30.14%
  Mean Pred: 1.075332
  Mean True: 1.076378

Agent 2:
  MAE:       0.259966
  Bias:      -0.003549
  % Error:   23.69%
  Mean Pred: 1.093700
  Mean True: 1.097249

Agent 3:
  MAE:       0.235311
  Bias:      -0.028035
  % Error:   19.02%
  Mean Pred: 1.209017
  Mean True: 1.237052

Overall MAE:     0.301931
Overall % Error: 25.79%
*** FAIL: >=10% error ***

Final models saved to outputs/
